In [ ]:
import pandas as pd
import numpy as np

vitalperiodic = pd.read_csv(r"C:\2)Projects\ICU_Patient_Predict_Risk_Analysis\dataset\vitalPeriodic.csv")

# shape

print("shape of file")
print(vitalperiodic.shape)
print("-"*100)

#info
print("info of file")
print(vitalperiodic.info())
print("-"*100)

# data types
print("datatypes of file")
print(vitalperiodic.dtypes)
print("-"*100)


# mssing values
print("missing values of file")
print(vitalperiodic.isnull().sum())


In [ ]:
# Column classification

id_cols = [
    "vitalperiodicid",
    "patientunitstayid"
]

time_cols = [
    "observationoffset"
]

vital_cols = [
    "temperature",
    "sao2",
    "heartrate",
    "respiration",
    "cvp",
    "etco2",
    "systemicsystolic",
    "systemicdiastolic",
    "systemicmean",
    "pasystolic",
    "padiastolic",
    "pamean",
    "st1",
    "st2",
    "st3",
    "icp"
]

print("ID columns:", id_cols)
print("Time column:", time_cols)
print("Vital measurement columns -> these are continuous:", vital_cols)

In [ ]:
# Number of unique ICU stays

print("Number of unique ICU stays",vitalperiodic["patientunitstayid"].nunique())

# Number of measurements per ICU stay

print("Number of measurements per ICU stay \n",vitalperiodic.groupby("patientunitstayid").size().describe())


# Observation time range

print("Observation time range \n ",vitalperiodic["observationoffset"].describe())

print("Minimum: \n", vitalperiodic["observationoffset"].min())
print("Maximum: \n ", vitalperiodic["observationoffset"].max())

In [ ]:
# vital columns:
vitalperiodic[vital_cols].describe().T

-> skewness 

In [ ]:
#Skewness

#Since this is numerical EDA, check the distribution shape.


vitalperiodic[vital_cols].skew().sort_values()

Outliers:
Outlier Analysis efficiently. Since you have 1.6 million rows, use counts instead of plotting all rows.

In [ ]:
# 7. Outlier Analysis using IQR

outlier_results = []

for col in vital_cols:
    Q1 = vitalperiodic[col].quantile(0.25)
    Q3 = vitalperiodic[col].quantile(0.75)
    IQR = Q3 - Q1

    lower = Q1 - 1.5 * IQR
    upper = Q3 + 1.5 * IQR

    count = ((vitalperiodic[col] < lower) | 
             (vitalperiodic[col] > upper)).sum()

    outlier_results.append([col, count])

outlier_df = pd.DataFrame(
    outlier_results,
    columns=["Variable", "Outlier_Count"]
)

outlier_df.sort_values("Outlier_Count", ascending=False)

Step 8 — Missing Value Analysis

In [ ]:
missing_vitals = vitalperiodic[vital_cols].isnull().mean() * 100

missing_vitals.sort_values(ascending=False).round(2)

Stpe 9:

Time Pattern EDA

This is especially important for vitalperiodic.csv because it is a time-series dataset.

We want to understand when measurements are recorded after ICU admission.

In [ ]:
vitalperiodic["observationoffset"].describe()

In [ ]:
(vitalperiodic["observationoffset"] < 0).sum()

In [ ]:
(vitalperiodic["observationoffset"] >= 0).sum()

In [ ]:
vitalperiodic["observationoffset"].isnull().sum()
print("Total rows:", len(vitalperiodic))
print("Negative:", (vitalperiodic["observationoffset"] < 0).sum())
print("Zero or positive:", (vitalperiodic["observationoffset"] >= 0).sum())
print("Missing:", vitalperiodic["observationoffset"].isna().sum())
print("Infinite:", (~np.isfinite(vitalperiodic["observationoffset"])).sum())

In [ ]:
vitalperiodic["observationoffset"].value_counts(dropna=False).head()

Meaning:

observationoffset
115    631
130    631
120    631
125    631
105    631
Name: count, dtype: int64

This means:

There are 631 measurement records with observationoffset = 115 minutes, 631 records at 130 minutes, etc.

It does not mean 631 patients.

CORRELATION:

Because this file has repeated measurements, we'll calculate correlation only among the 16 vital measurement columns, not the ID/time columns.

In [ ]:
# Correlation between vital measurements

corr = vitalperiodic[vital_cols].corr()

corr

Note :You don't need to manually inspect the huge matrix. Use the same classification approach we used for
       patient.csv

In [ ]:
# Classify correlation strength

results = []

for i in range(len(corr.columns)):
    for j in range(i + 1, len(corr.columns)):

        value = corr.iloc[i, j]

        if abs(value) < 0.3:
            category = "Weak"
        elif abs(value) < 0.7:
            category = "Moderate"
        else:
            category = "Strong"

        results.append([
            corr.columns[i],
            corr.columns[j],
            round(value, 3),
            category
        ])

corr_results = pd.DataFrame(
    results,
    columns=["Feature_1", "Feature_2", "Correlation", "Category"]
)

corr_results

In [ ]:
corr_results[corr_results["Category"] != "Weak"]

#Modified code :

added point

Your correlation analysis is mostly correct, but there is one issue: the rows with NaN correlation should not be classified as Strong. This happens because abs(np.nan) >= 0.7 evaluates as false, so your else incorrectly labels it "Strong".

In [ ]:
results = []

for i in range(len(corr.columns)):
    for j in range(i + 1, len(corr.columns)):

        value = corr.iloc[i, j]

        if pd.isna(value):
            category = "Not Available"
        elif abs(value) < 0.3:
            category = "Weak"
        elif abs(value) < 0.7:
            category = "Moderate"
        else:
            category = "Strong"

        results.append([
            corr.columns[i],
            corr.columns[j],
            round(value, 3) if pd.notna(value) else value,
            category
        ])

corr_results = pd.DataFrame(
    results,
    columns=["Feature_1", "Feature_2", "Correlation", "Category"]
)

In [ ]:
corr_results[corr_results["Category"] != "Weak"]

Step 10 — Potential leakage:

Now we do the last part of vitalperiodic.csv EDA: EDA Findings and Leakage Check.

One important point first:

Target is not in vitalperiodic.csv

Your target is:

hospitaldischargestatus

That column exists in patient.csv, not in vitalperiodic.csv.

Therefore, we cannot check target leakage directly inside vitalperiodic.csv alone.

For this file, we only need to check whether any columns contain information that would obviously reveal the outcome or future information. Your columns are mainly IDs, time, and vital measurements, so there is no obvious outcome column.

